# Combining Spatial Enhancement Methods
**Bone Scan Image — Full Pipeline (a) → (h)**

| Step | Description |
|------|-------------|
| **(a)** | Original bone scan image |
| **(b)** | Laplacian of (a) — sharpens edges |
| **(c)** | (a) + (b) — basic sharpened image |
| **(d)** | Sobel gradient magnitude of (a) |
| **(e)** | Sobel image (d) smoothed with 5×5 box filter |
| **(f)** | Mask = (b) × (e) — product of Laplacian and smoothed Sobel |
| **(g)** | (a) + (f) — sharpened image using mask |
| **(h)** | Power-law (gamma) transformation applied to (g) — final result |

## 1. Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, uniform_filter
from skimage import io, img_as_float, filters
from skimage.color import rgb2gray
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Upload Bone Scan Image

In [ ]:
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image_raw = io.imread(filename)

print(f'Loaded : {filename}')
print(f'Shape  : {image_raw.shape}')
print(f'Dtype  : {image_raw.dtype}')

## 3. Run Full Pipeline (a) → (h)

In [ ]:
# ── Parameters ──────────────────────────────────────────
sigma = 3         # Gaussian sigma (not used in this pipeline but kept for reference)
gamma = 0.4       # <-- adjust (try 0.3, 0.4, 0.5)
c     = 1.0
# ────────────────────────────────────────────────────────

# (a) Original
if image_raw.ndim == 3:
    img_a = rgb2gray(img_as_float(image_raw))
else:
    img_a = img_as_float(image_raw)

# (b) Laplacian
laplacian_kernel = np.array([[1,  1, 1],
                              [1, -8, 1],
                              [1,  1, 1]], dtype=np.float64)
img_b_raw     = convolve(img_a, laplacian_kernel)
img_b_display = (img_b_raw - img_b_raw.min()) / (img_b_raw.max() - img_b_raw.min())

# (c) Basic sharpen: (a) + (b)
img_c = np.clip(img_a + img_b_raw, 0, 1)

# (d) Sobel gradient
img_d = filters.sobel(img_a)

# (e) Sobel smoothed with 5x5 box filter
img_e = uniform_filter(img_d, size=5)

# (f) Mask = (b) x (e)
img_f_raw     = img_b_raw * img_e
img_f_display = (img_f_raw - img_f_raw.min()) / (img_f_raw.max() - img_f_raw.min())

# (g) (a) + (f)
img_g = np.clip(img_a + img_f_display, 0, 1)

# (h) Power-law on (g)
img_h = np.clip(c * np.power(img_g, gamma), 0, 1)

print('Pipeline complete!')
print(f'  gamma = {gamma}')
print(f'  (h) range: [{img_h.min():.4f}, {img_h.max():.4f}]')

## 4. Full Pipeline — All Steps (a) to (h)

In [ ]:
steps = [
    (img_a,         '(a) Original'),
    (img_b_display, '(b) Laplacian'),
    (img_c,         '(c) (a) + (b)'),
    (img_d,         '(d) Sobel'),
    (img_e,         '(e) Sobel + 5×5 Box'),
    (img_f_display, '(f) Mask: (b)×(e)'),
    (img_g,         '(g) (a) + (f)'),
    (img_h,         f'(h) Power-Law  γ={gamma}'),
]

fig, axes = plt.subplots(2, 4, figsize=(22, 13))
fig.suptitle('Combining Spatial Enhancement Methods — Full Pipeline',
             fontsize=15, fontweight='bold', y=1.01)

for ax, (img, label) in zip(axes.flatten(), steps):
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_xlabel(label, fontsize=12, labelpad=8)

plt.subplots_adjust(wspace=0.04, hspace=0.12, bottom=0.07)
plt.savefig('result_full_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result_full_pipeline.png')

## 5. Compare (a), (g), (h) — as the exam requires

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 8))
fig.suptitle('Compare: Original  |  Sharpened  |  Power-Law',
             fontsize=14, fontweight='bold', y=1.01)

compare = [
    (img_a, '(a) Original'),
    (img_g, '(g) Sharpened: (a)+(f)'),
    (img_h, f'(h) Power-Law  γ={gamma}'),
]

for ax, (img, label) in zip(axes, compare):
    ax.imshow(img, cmap='gray')
    ax.axis('off')
    ax.set_xlabel(label, fontsize=12, labelpad=8)

plt.subplots_adjust(wspace=0.05, bottom=0.1)
plt.savefig('result_comparison_a_g_h.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: result_comparison_a_g_h.png')

## 6. Compare Different Gamma Values

In [ ]:
gammas = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5]

fig, axes = plt.subplots(1, len(gammas), figsize=(24, 6))
fig.suptitle('Effect of Different Gamma Values on (g)',
             fontsize=13, y=1.01)

for ax, g in zip(axes, gammas):
    out = np.clip(np.power(img_g, g), 0, 1)
    ax.imshow(out, cmap='gray')
    ax.axis('off')
    ax.set_xlabel(f'γ = {g}', fontsize=11, labelpad=6)

plt.subplots_adjust(wspace=0.04, bottom=0.1)
plt.savefig('result_gamma_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Download Results

In [ ]:
files.download('result_full_pipeline.png')
files.download('result_comparison_a_g_h.png')
files.download('result_gamma_comparison.png')
print('Done! All results downloaded.')